# VisionQC PET — Ron 88 ilk model (v1)

**Kullanım:** Çalışma zamanı → Çalışma zamanı türünü değiştir → en hızlı GPU: **A100** (yoksa **L4**, en son **T4**). Sonra **Çalışma zamanı → Tümünü çalıştır**.
Başta Google Drive izni isteyecek (bir kez onayla). Zip'i önceden Drive'a koymak, 334 MB'ı Colab'a elle yüklemekten çok daha hızlı ve güvenlidir. Gerisi kendiliğinden akar; eğitim ilerlemesi 8. hücrede canlı görünür (T4'te yaklaşık 30–60 dk; A100/L4'te belirgin daha kısa).

**Ne yapar** (veri denetim raporundaki plan):

- Veri seti: `jihadakbr/water-bottle-defect-level-detection-dataset` (Kaggle'dan indirdiğin zip). Zip'i Colab'ın sol dosya panelinde `/content/dataset.zip` olarak bırakman ya da **Google Drive'ının ana klasörüne** koyman yeterli; not defteri onu kendisi bulup açar. Bulamazsa yükleme penceresi açar.
- Havuz: yalnızca **sağlam / düşük dolum / kapaksız / gevşek kapak** görüntüleri (1.150). Havuz dosya adından değil **etiketlerden** türetilir: `bottle_ron88` var, yabancı madde / hasarlı etiket / farklı marka yok, en fazla bir kusur sınıfı.
- Sınıflar: `bottle`, `underfill`, `missing_cap`, `loose_cap`. Kalan 350 görüntü kullanılmaz.
- Eğitim/doğrulama/test **rastgele değil**, her grupta zaman bloklarına göre ayrılır; blok sınırlarında tampon bölge atılır.
- Sonunda kutu metrikleri (mAP) ve görüntü düzeyinde "sağlam / kusurlu" tablosu üretilir. Her şey Drive'a (`visionqc_ron88`) kaydedilir.

**Önemli:** Bu bölme yakın çekim sızıntısını azaltır ama fiziksel şişe bağımsızlığı sağlamaz. Buradaki test sonucu "aynı çekim ortamında" sonuçtur. Gerçek genelleme için en alttaki hücreyle kendi şişelerinden, eğitimde kullanılmamış küçük bir grubu dene.

In [ ]:
# 1) AYARLAR (varsayılanlar hazır; değiştirmen gerekmez)
from pathlib import Path

# --- Veri ---
SOURCE = "auto"                         # "auto": Drive'daki zip'i bul, yoksa yükleme penceresi aç | "kaggle": otomatik indir
KAGGLE_DATASET = "jihadakbr/water-bottle-defect-level-detection-dataset"
DRIVE_ROOT = "/content/drive/MyDrive"
DRIVE_ZIP = ""                          # boş bırak: önce /content/*.zip, sonra DRIVE_ROOT (+1 alt klasör) içinde >100 MB zip aranır; istersen tam yolu yaz

# --- Çıktı ---
WORK = Path("/content")
SAVE_TO_DRIVE = True
DRIVE_OUT = "/content/drive/MyDrive/visionqc_ron88"

# --- Sınıf şeması ---
NEW_NAMES = ["bottle", "underfill", "missing_cap", "loose_cap"]
CLASS_MAP = {                           # kaynak sınıf adı -> yeni sınıf adı
    "bottle_ron88":     "bottle",
    "defect_low_fill":  "underfill",
    "defect_no_cap":    "missing_cap",
    "defect_loose_cap": "loose_cap",
}
POOL = {"good": 600, "underfill": 250, "missing_cap": 150, "loose_cap": 150}   # rapordaki beklenen sayılar

# --- Zaman blokları bölmesi ---
K_BLOCKS = 10                           # her grup K ardışık zaman bloğuna bölünür
BLOCK_PATTERN = ["train", "val", "train", "train", "test",
                 "train", "train", "val", "train", "test"]   # 6 train / 2 val / 2 test blok
BUFFER_SEC = 10                         # farklı bölümler arası en az boşluk (saniye)
BUFFER_FRAMES = 5                       # zaman damgası okunamazsa: boşluk (kare sayısı)

# --- Eğitim ---
MODEL = "yolo11s.pt"
IMGSZ = 960                             # bellek/zaman sorunu olursa 640
EPOCHS = 80
BATCH = None                          # None: GPU belleğine göre otomatik (A100 32 / L4 24 / T4 16)
PATIENCE = 20
SEED = 42
CONF = 0.25                             # görüntü düzeyi karar eşiği

In [ ]:
# 2) GPU kontrolü, kurulum, Drive
import subprocess, sys
import torch
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU seç, sonra baştan çalıştır"
gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {gpu_name} ({gpu_gb:.0f} GB)")
if BATCH is None:
    BATCH = 32 if gpu_gb >= 35 else 24 if gpu_gb >= 20 else 16
import os
WORKERS = min(8, os.cpu_count() or 2)
print(f"batch={BATCH}, workers={WORKERS}")
if "T4" in gpu_name:
    print("Not: T4 en yavaş seçenek. Çalışma zamanı > Çalışma zamanı türünü değiştir bölümünden A100 (yoksa L4) seçersen eğitim belirgin hızlanır.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "kagglehub"], check=True)
import ultralytics
print("ultralytics", ultralytics.__version__)

if SAVE_TO_DRIVE or SOURCE in ("auto", "drive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
    except Exception as e:
        # İzin penceresi tamamlanmadıysa / engellendiyse eğitim yine de devam eder, çıktılar yerelde kalır
        print("UYARI: Drive bağlanamadı:", repr(e)[:150])
        print("Devam ediliyor: çıktılar yalnızca /content altına yazılacak (oturum kapanınca silinir; en sonda indirme verilecek).")
        SAVE_TO_DRIVE = False
RUNS = (Path(DRIVE_OUT) / "runs") if SAVE_TO_DRIVE else (WORK / "runs")   # eğitim çıktıları Drive'a yazılır: bağlantı kopsa da kaybolmaz
print("Eğitim çıktı klasörü:", RUNS)

In [ ]:
# 3) Veriyi al
import os, zipfile, shutil

ZIPS = WORK / "zips"; ZIPS.mkdir(exist_ok=True)

def extract(zip_path):
    dst = WORK / "raw"; dst.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dst)
    return dst

def find_zip():
    if DRIVE_ZIP:
        return Path(DRIVE_ZIP) if Path(DRIVE_ZIP).exists() else None
    local = [p for p in WORK.glob("*.zip") if p.stat().st_size > 100e6]       # Colab sol paneline yüklenen zip (/content/dataset.zip)
    if local:
        pick = max(local, key=lambda p: p.stat().st_mtime)
        print("Colab dosyalarında bulunan zip:", pick)
        return pick
    root = Path(DRIVE_ROOT)
    cands = [p for p in list(root.glob("*.zip")) + list(root.glob("*/*.zip")) if p.stat().st_size > 100e6]
    named = [p for p in cands if re.search(r"bottle|water|ron|archive", p.name, re.I)]
    cands = named or cands
    if not cands:
        return None
    pick = max(cands, key=lambda p: p.stat().st_mtime)
    print("Drive'da bulunan zip:", [str(p) for p in cands], "-> kullanılan:", pick)
    return pick

import re
if SOURCE == "kaggle":
    try:
        import kagglehub
        RAW = Path(kagglehub.dataset_download(KAGGLE_DATASET))      # herkese açık veri: giriş gerekmeyebilir
        print("kagglehub ile indirildi:", RAW)
    except Exception as e:
        print("kagglehub olmadı (", repr(e)[:160], ") -> Kaggle API'ye geçiliyor")
        if "KAGGLE_USERNAME" not in os.environ and not (Path.home() / ".kaggle" / "kaggle.json").exists():
            try:
                from google.colab import userdata
                os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
                os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
            except Exception:
                from google.colab import files
                print("kaggle.json dosyanı yükle (kaggle.com > Settings > API > Create New Token):")
                up = files.upload()
                kd = Path.home() / ".kaggle"; kd.mkdir(exist_ok=True)
                (kd / "kaggle.json").write_bytes(next(iter(up.values())))
                os.chmod(kd / "kaggle.json", 0o600)
        subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(ZIPS)], check=True)
        RAW = extract(next(ZIPS.glob("*.zip")))
else:
    zp = find_zip()
    if zp is None:
        from google.colab import files
        print("Drive'da zip bulunamadı. Zip dosyasını şimdi yükle (334 MB; yüklemesi birkaç dakika sürebilir).")
        print("İpucu: bir dahaki sefer zip'i Drive'ının ana klasörüne koyarsan bu adım atlanır.")
        up = files.upload()
        zp = Path(next(iter(up)))
    RAW = extract(zp)
    print("Açıldı:", zp)

n_img = sum(1 for p in RAW.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
n_txt = sum(1 for p in RAW.rglob("*.txt"))
print(f"{n_img} görüntü, {n_txt} txt  ({RAW})")
for p in sorted(RAW.iterdir())[:10]:
    print("  ", p.name)

In [ ]:
# 4) Dosyaları indeksle, etiketlerden havuzu türet, zaman damgalarını oku
import re, collections, yaml
import numpy as np
import pandas as pd
from datetime import datetime

imgs = sorted(p for p in RAW.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
dups = [s for s, c in collections.Counter(p.stem for p in imgs).items() if c > 1]
assert not dups, f"Aynı adlı görüntüler var (örn. {dups[:3]}); klasörler arası ad çakışması."
txts = {p.stem: p for p in RAW.rglob("*.txt") if p.name.lower() != "classes.txt"}

yamls = list(RAW.rglob("data.yaml"))
assert yamls, "data.yaml bulunamadı"
orig_names = yaml.safe_load(yamls[0].read_text())["names"]
if isinstance(orig_names, list):
    orig_names = dict(enumerate(orig_names))
print("Kaynak sınıflar:", orig_names)
missing = set(CLASS_MAP) - set(orig_names.values())
assert not missing, f"CLASS_MAP'teki sınıflar data.yaml'da yok: {missing}. 1. hücredeki CLASS_MAP anahtarlarını yukarıdaki adlara göre düzelt."
orig_id2new = {i: NEW_NAMES.index(CLASS_MAP[n]) for i, n in orig_names.items() if n in CLASS_MAP}
DEF_IDS = {1, 2, 3}

problems, recs = [], []
for p in imgs:
    lab = txts.get(p.stem)
    rec = dict(path=p, fname=p.name, lines=[], classes=set(), has_other=False, ok=True)
    if lab is None:
        problems.append(f"etiket yok: {p.name}"); rec["ok"] = False; recs.append(rec); continue
    for ln in lab.read_text().splitlines():
        parts = ln.split()
        if not parts:
            continue
        try:
            cid = int(float(parts[0])); vals = [float(v) for v in parts[1:]]
            assert len(vals) == 4 and all(0 <= v <= 1 for v in vals)
        except Exception:
            problems.append(f"bozuk satır: {lab.name}: {ln!r}"); rec["ok"] = False; continue
        if cid in orig_id2new:
            rec["lines"].append(f"{orig_id2new[cid]} " + " ".join(parts[1:]))
        else:
            rec["has_other"] = True            # debris / label_damage / other_brand
    rec["classes"] = {int(l.split()[0]) for l in rec["lines"]}
    recs.append(rec)
print("Etiket sorunu:", len(problems)); print(*problems[:10], sep="\n")

# havuz kuralı: şişe var, şema dışı sınıf yok, en fazla bir kusur sınıfı
why = collections.Counter(); pool = []
for r in recs:
    d = r["classes"] & DEF_IDS
    if not r["ok"]:            why["etiket sorunu"] += 1
    elif r["has_other"]:       why["yabancı madde / hasarlı etiket / farklı marka"] += 1
    elif 0 not in r["classes"]: why["ron88 şişesi yok"] += 1
    elif len(d) > 1:           why["birden çok kusur"] += 1
    else:
        r["group"] = "good" if not d else NEW_NAMES[next(iter(d))]
        pool.append(r)
print("\nHavuz dışı:", dict(why))
counts = collections.Counter(r["group"] for r in pool)
print("Havuz:", dict(counts), "| beklenen:", POOL)
if dict(counts) != POOL:
    print("UYARI: sayılar rapordakinden farklı. Etiket kuralı farklı sonuç veriyor olabilir; yine de devam ediliyor.")
assert all(counts[g] >= 20 for g in POOL), "Bir grupta çok az görüntü var; yukarıdaki çıktıya bak."

n_good_name = sum("good" in r["fname"].lower() for r in pool if r["group"] == "good")
print(f"Bilgi: 'good' grubundaki {counts['good']} görüntünün {n_good_name}'inde dosya adında 'good' geçiyor.")
print("Örnek dosya adları:", [r["fname"] for r in pool[::250]])

def parse_ts(name):
    m = re.search(r"(20\d{2})\D?(\d{2})\D?(\d{2})\D{0,3}(\d{2})\D?(\d{2})\D?(\d{2})", name)
    if m:
        try:
            return datetime(*map(int, m.groups())).timestamp()
        except ValueError:
            pass
    m = re.search(r"(?<!\d)(1[6-9]\d{8})(\d{3})?(?!\d)", name)       # epoch saniye / ms
    if m:
        return int(m.group(1)) + (int(m.group(2)) / 1000 if m.group(2) else 0)
    return None

for r in pool:
    r["ts"] = parse_ts(r["fname"])
if all(r["ts"] is not None for r in pool):
    TS_MODE, BUF, UNIT = "time", BUFFER_SEC, "sn"
    print("\nZaman damgası dosya adından okundu.")
else:
    bad = [r["fname"] for r in pool if r["ts"] is None]
    print(f"\nUYARI: {len(bad)} dosyada zaman damgası okunamadı (örn. {bad[:3]}).")
    print("Yedek mod: her grupta dosya adı sırası (doğal sıralama) zaman yerine kullanılıyor; tampon kare cinsinden.")
    natkey = lambda s: [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]
    for g in POOL:
        for i, r in enumerate(sorted((x for x in pool if x["group"] == g), key=lambda x: natkey(x["fname"]))):
            r["ts"] = float(i)
    TS_MODE, BUF, UNIT = "index", BUFFER_FRAMES, "kare"

for g in POOL:
    t = sorted(r["ts"] for r in pool if r["group"] == g)
    print(f"  {g:12s} n={len(t):4d}  aralık={t[-1]-t[0]:9.0f} {UNIT}  medyan adım={np.median(np.diff(t)):.1f} {UNIT}")

In [ ]:
# 5) Zaman blokları bölmesi + YOLO veri seti yazımı
def split_group(items):
    items.sort(key=lambda r: (r["ts"], r["fname"]))
    n, k = len(items), K_BLOCKS
    for i, r in enumerate(items):
        r["block"] = min(i * k // n, k - 1)
        r["split"] = BLOCK_PATTERN[r["block"]]
    for b in range(1, k):
        if BLOCK_PATTERN[b] == BLOCK_PATTERN[b - 1]:
            continue
        last = max(r["ts"] for r in items if r["block"] == b - 1)
        first = min(r["ts"] for r in items if r["block"] == b)
        mid = (last + first) / 2
        for r in items:
            if r["block"] in (b - 1, b) and abs(r["ts"] - mid) < BUF / 2:
                r["split"] = None

assert len(BLOCK_PATTERN) == K_BLOCKS
for g in POOL:
    split_group([r for r in pool if r["group"] == g])

kept = [r for r in pool if r["split"]]
print(f"Tampon nedeniyle atılan: {len(pool) - len(kept)} / {len(pool)}")

DS = WORK / "ds"
if DS.exists():
    shutil.rmtree(DS)
for s in ("train", "val", "test"):
    (DS / "images" / s).mkdir(parents=True); (DS / "labels" / s).mkdir(parents=True)
for r in kept:
    shutil.copy2(r["path"], DS / "images" / r["split"] / r["fname"])
    (DS / "labels" / r["split"] / (r["path"].stem + ".txt")).write_text("\n".join(r["lines"]) + "\n")
(DS / "data.yaml").write_text(yaml.safe_dump(
    {"path": str(DS), "train": "images/train", "val": "images/val", "test": "images/test",
     "names": dict(enumerate(NEW_NAMES))}, sort_keys=False))

man = pd.DataFrame([dict(fname=r["fname"], group=r["group"], ts=r["ts"], block=r["block"], split=r["split"],
                         classes=",".join(NEW_NAMES[i] for i in sorted(r["classes"]))) for r in pool])
man.to_csv(WORK / "split_manifest.csv", index=False)

print("\nGörüntü sayısı (bölüm x grup):"); print(pd.crosstab(man["split"].fillna("atıldı"), man["group"]))
box_counts = pd.DataFrame([{"split": r["split"], **{n: sum(1 for l in r["lines"] if int(l.split()[0]) == i) for i, n in enumerate(NEW_NAMES)}} for r in kept]).groupby("split").sum()
print("\nKutu sayısı (bölüm x sınıf):"); print(box_counts)
for s in ("train", "val", "test"):
    assert (box_counts.loc[s] > 0).all(), f"{s} bölümünde eksik sınıf var; BLOCK_PATTERN/BUFFER_SEC'i gözden geçir"

In [ ]:
# 6) Sızıntı tanısı: val/test görüntüleri eğitime ne kadar yakın?
from PIL import Image

def dhash_bits(path):
    a = np.asarray(Image.open(path).convert("L").resize((9, 8), Image.LANCZOS), dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()

by_split = {s: [r for r in kept if r["split"] == s] for s in ("train", "val", "test")}
H = {s: np.stack([dhash_bits(DS / "images" / s / r["fname"]) for r in rs]) for s, rs in by_split.items()}
T = {s: np.array([r["ts"] for r in rs]) for s, rs in by_split.items()}
G = {s: np.array([r["group"] for r in rs]) for s, rs in by_split.items()}

for s in ("val", "test"):
    ham = (H[s][:, None, :] != H["train"][None, :, :]).sum(-1).min(1)
    near_t = np.array([np.abs(T["train"][G["train"] == g] - t).min() for t, g in zip(T[s], G[s])])
    print(f"{s:4s} n={len(ham):3d} | dHash <=2 bit eğitime yakın: {(ham <= 2).sum():3d} | eğitim çekimine <=5 {UNIT}: {(near_t <= 5).sum():3d} | en yakın eğitim çekimi medyan: {np.median(near_t):.0f} {UNIT}")
print("\nReferans (orijinal rastgele bölme): doğrulamanın 295/300'ü dHash<=2, 294/300'ü <=5 sn.")
print("Burada belirgin düşük olmalı; sabit arka plan dHash'i baskıladığı için sıfır olmayabilir.")

In [ ]:
# 7) Etiket kontrolü: her grup için 3 örnek (kutular çizili)
import random
import matplotlib.pyplot as plt
from PIL import ImageDraw

rng = random.Random(0)
COLORS = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e"]
fig, axes = plt.subplots(3, 4, figsize=(22, 10))
for col, g in enumerate(POOL):
    cand = [r for r in kept if r["group"] == g]
    for row, r in enumerate(rng.sample(cand, 3)):
        im = Image.open(DS / "images" / r["split"] / r["fname"]).convert("RGB")
        d = ImageDraw.Draw(im); W, Hh = im.size
        for l in r["lines"]:
            c, x, y, w, h = l.split(); c = int(c); x, y, w, h = (float(x) * W, float(y) * Hh, float(w) * W, float(h) * Hh)
            d.rectangle([x - w / 2, y - h / 2, x + w / 2, y + h / 2], outline=COLORS[c], width=5)
            d.text((x - w / 2 + 6, y - h / 2 + 6), NEW_NAMES[c], fill=COLORS[c])
        axes[row, col].imshow(im); axes[row, col].axis("off"); axes[row, col].set_title(f"{g} · {r['split']}", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# 8) EĞİTİM (canlı ilerleme burada görünür)
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data=str(DS / "data.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
    seed=SEED, deterministic=True, workers=WORKERS, cache="ram",
    project=str(RUNS), name="ron88_v1", exist_ok=True,
    # sabit arka plana aşırı uymayı azaltmak için renk/ölçek çeşitliliği
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, scale=0.5, translate=0.1, fliplr=0.5, mosaic=1.0, close_mosaic=10,
)
RUN_DIR = Path(model.trainer.save_dir)      # ultralytics sürümüne göre klasör adı değişebilir
BEST = RUN_DIR / "weights" / "best.pt"
print("En iyi ağırlık:", BEST, BEST.exists())

In [ ]:
# 9) Değerlendirme: kutu metrikleri (val ve test)
best = YOLO(str(BEST))

def per_class(m):
    rows = [dict(sınıf=NEW_NAMES[ci], P=m.box.p[i], R=m.box.r[i], AP50=m.box.ap50[i], AP50_95=m.box.ap[i])
            for i, ci in enumerate(m.box.ap_class_index)]
    return pd.DataFrame(rows).round(3)

results, eval_dirs = {}, {}
for s in ("val", "test"):
    m = best.val(data=str(DS / "data.yaml"), split=s, imgsz=IMGSZ, plots=True,
                 project=str(RUNS), name=f"eval_{s}", exist_ok=True, verbose=False)
    results[s] = per_class(m)
    eval_dirs[s] = Path(m.save_dir)
    print(f"\n=== {s.upper()} === mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}")
    print(results[s].to_string(index=False))
print("\nNot: best.pt epoch seçimi val ile yapıldı, yani val metriği hafif iyimser. Raporlanacak sayı TEST olmalı.")

fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for ax, (title, f) in zip(axes, [("Eğitim eğrileri", RUN_DIR / "results.png"),
                                 ("Test hata matrisi", eval_dirs["test"] / "confusion_matrix_normalized.png"),
                                 ("Test PR eğrisi", eval_dirs["test"] / "BoxPR_curve.png")]):
    ax.axis("off"); ax.set_title(title)
    if f.exists():
        ax.imshow(plt.imread(str(f)))
plt.tight_layout(); plt.show()

In [ ]:
# 10) Görüntü düzeyi karar: şişe "sağlam" mı "kusurlu" mu? (test bölümü)
test_recs = by_split["test"]
paths = [str(DS / "images" / "test" / r["fname"]) for r in test_recs]
rows = []
for r, out in zip(test_recs, best.predict(paths, conf=CONF, imgsz=IMGSZ, stream=True, verbose=False)):
    pred = {int(c) for c in out.boxes.cls.tolist()}
    rows.append(dict(group=r["group"], gt_defect=bool(r["classes"] & DEF_IDS), pred_defect=bool(pred & DEF_IDS),
                     gt_cls=r["classes"] & DEF_IDS, pred_cls=pred & DEF_IDS, bottle_found=0 in pred))
df = pd.DataFrame(rows)

print(f"conf={CONF}")
print(pd.crosstab(df["gt_defect"].map({False: "gerçek: sağlam", True: "gerçek: kusurlu"}),
                  df["pred_defect"].map({False: "tahmin: sağlam", True: "tahmin: kusurlu"})))
good = df[~df.gt_defect]
print(f"\nSağlamı kusurlu sanma (yanlış alarm): {good.pred_defect.mean():.3f}  (n={len(good)})")
for g in ("underfill", "missing_cap", "loose_cap"):
    sub = df[df.group == g]; wid = NEW_NAMES.index(g)
    print(f"{g:12s} n={len(sub):3d}  doğru kusur sınıfı: {sub.pred_cls.map(lambda s: wid in s).mean():.3f}  hiç kusur yakalanmadı: {(~sub.pred_defect).mean():.3f}")
print(f"Şişe bulunamayan görüntü oranı: {(~df.bottle_found).mean():.3f}")
df.drop(columns=["gt_cls", "pred_cls"]).to_csv(WORK / "test_image_level.csv", index=False)

In [ ]:
# 11) Kaydet (ağırlıklar, bölme listesi, sonuçlar)
out = WORK / "ron88_v1_export"; out.mkdir(exist_ok=True)
shutil.copy2(BEST, out / "best.pt")
shutil.copy2(WORK / "split_manifest.csv", out / "split_manifest.csv")
shutil.copy2(WORK / "test_image_level.csv", out / "test_image_level.csv")
for s, d in results.items():
    d.to_csv(out / f"per_class_{s}.csv", index=False)
for f in RUN_DIR.glob("*.png"):
    shutil.copy2(f, out / f"train_{f.name}")
for s, d in eval_dirs.items():
    for f in d.glob("*.png"):
        shutil.copy2(f, out / f"{s}_{f.name}")
if SAVE_TO_DRIVE:
    shutil.copytree(out, Path(DRIVE_OUT) / "export", dirs_exist_ok=True)
    print("Drive'a kaydedildi:", Path(DRIVE_OUT) / "export")
print(sorted(p.name for p in out.iterdir()))
if not SAVE_TO_DRIVE:
    z = shutil.make_archive(str(WORK / "ron88_v1_export"), "zip", out)
    print("Drive yok: sonuçlar zip'lendi ->", z)
    try:
        from google.colab import files
        files.download(z)
    except Exception as e:
        print("Otomatik indirme olmadı; sol panelden", z, "dosyasını indir.", repr(e)[:100])

## Yeni ortama genelleme kontrolü (kendi şişelerinin)

Aşağıdaki hücre, eğitimde **hiç kullanılmamış** kendi fotoğraflarını modele verir. Drive'da `visionqc_ron88/own_test` klasörü oluşturup farklı arka plan/ışıkta 20–40 fotoğraf (sağlam, kapaksız, gevşek kapak, düşük dolum) koy ve hücreyi çalıştır. Sonuç bu bölmedeki test sonucundan çok düşükse, model arka plana/çekim koşuluna uymuş demektir; bu durumda ikinci bir su şişesi kaynağıyla veri çeşitlendirilmelidir.

In [ ]:
# 12) Kendi fotoğrafların üzerinde dene
OWN_DIR = Path(DRIVE_OUT) / "own_test"
own = sorted(p for p in OWN_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}) if OWN_DIR.exists() else []
print(len(own), "fotoğraf")
if own:
    res = best.predict([str(p) for p in own], conf=CONF, imgsz=IMGSZ, verbose=False)
    fig, axes = plt.subplots(3, 4, figsize=(22, 10))
    for ax in axes.flat:
        ax.axis("off")
    for ax, p, r in zip(axes.flat, own, res):
        ax.imshow(r.plot()[..., ::-1]); ax.set_title(p.name, fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print(f"{OWN_DIR} klasörünü oluşturup fotoğrafları koy, sonra bu hücreyi tekrar çalıştır.")